# Notebook for internal train / val splits creation

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
train_data = pd.read_csv('<CT_RATE_DATASET_DIR>/train_labels.csv')

# read in list of volumnes to ignore from txt file
with open('<CT_RATE_DATASET_DIR>/metadata/no_chest_train.txt') as f:
    no_chest_train = f.read().splitlines()

In [ ]:
# Remove rows with VolumeNames not in the no_chest_train list
len(no_chest_train)
train_data = train_data[~train_data['VolumeName'].isin(no_chest_train)]

In [35]:
# Filenames are in format: train_X_a_1.nii.gz, get train_X from the VolumeName column
train_folder_id = train_data['VolumeName'].apply(lambda x: x.split('_')[1]).drop_duplicates().values
train_folder_id
# Split the train_folder_id (i.e. patient split) into internal train and validation sets (85% train, 15% val)
# Split at the patient level to avoid data leakage
train_patient_ids, val_patients_ids = train_test_split(train_folder_id, test_size=0.15, random_state=42)
print('Number of patients in internal train set:', len(train_patient_ids))
print('Number of patients in internal val set:', len(val_patients_ids))

# Get the internal train and validation csv files
internal_train_csv = train_data[train_data['VolumeName'].apply(lambda x: x.split('_')[1]).isin(train_patient_ids)]
internal_val_csv = train_data[train_data['VolumeName'].apply(lambda x: x.split('_')[1]).isin(val_patients_ids)]
print('Number of volumes in internal train set:', len(internal_train_csv))
print('Number of volumes in internal val set:', len(internal_val_csv))

Number of patients in internal train set: 17000
Number of patients in internal val set: 3000
Number of volumes in internal train set: 40029
Number of volumes in internal val set: 7120


In [ ]:
# Save the list of VolumeNames in the internal train and validation sets as .txt files
with open('<CT_RATE_DATASET_DIR>/labels/internal_train_volumes.txt', 'w') as f:
    for item in internal_train_csv['VolumeName'].values:
        f.write("%s\n" % item)


with open('<CT_RATE_DATASET_DIR>/labels/internal_val_volumes.txt', 'w') as f:
    for item in internal_val_csv['VolumeName'].values:
        f.write("%s\n" % item)


In [ ]:
# Save teh internal train and validation csv files
internal_train_csv.to_csv('<CT_RATE_DATASET_DIR>/labels/internal_train_labels.csv', index=False)
internal_val_csv.to_csv('<CT_RATE_DATASET_DIR>/labels/internal_val_labels.csv', index=False)